# 문제 2-4 : 여행 계획 분석기

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0)

schemas = [
    ResponseSchema(name="destination", description="여행지 (도시 또는 지역 이름)"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="여행 예산 또는 총 지출 (예: 30만원)"),
    ResponseSchema(name="rating", description="여행 추천도 (1~5 사이의 숫자)"),
    ResponseSchema(name="activities", description="여행에서 한 주요 활동 목록", type="list"),
]
parser = StructuredOutputParser.from_response_schemas(schemas)

pt = PromptTemplate(
    template="다음 여행 후기(또는 계획)에서 정보를 추출하세요.\n\n{fmt}\n\n텍스트: {text}",
    input_variables=["text"],
    partial_variables={"fmt": parser.get_format_instructions()},
)

chain = pt | llm | parser

text = input("여행 후기나 계획을 입력하세요 (엔터만 누르면 예시 실행): ").strip() or (
    "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, "
    "자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. "
    "5점 만점에 4점 정도 줄 수 있을 것 같아요."
)

try:
    r = chain.invoke({"text": text})
    print(type(r))
    print(json.dumps(r, ensure_ascii=False, indent=4))
except Exception as e:
    print("오류:", e)


<class 'dict'>
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": [
        "해운대에서 바다구경",
        "자갈치시장에서 회 먹기",
        "감천문화마을 구경"
    ]
}
